# Magnitude equation — conditional OLS

Given an arbitrage is open, *how large* is the round-trip gap? OLS on the log gap size, conditional
on existence (`D == 1`):

$$
\log \mathrm{Gap}_{p,t}=\alpha_p
+\theta_1\,\log(1+\mathrm{Gap}_{p,t-1})+\theta_2\,\log(\text{base\_fee}_t)+\theta_3\,\text{gas\_util}_{t-1}
+\theta_4\,\log(1+\text{tip\_p90}_{t-1})+\theta_5\,\overline{\log(1+\text{mev})}_{p,t-1}
+\theta_6\,\overline{\text{nb\_swaps\_ewma}}_{p,t-1}+\theta_7\,\log(\text{ewma\_vol}_t)
+\varepsilon_{p,t}\qquad\text{s.t. } D_{p,t}=1
$$

Same panel and lag discipline as the existence equation, but OLS. The lagged gap enters as
`log(1+Gap_{t-1})` since it can be exactly 0 (a freshly-opened spell). All logic lives in
`arblib.estimation`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd

from arblib import estimation as est
from arblib.config import STUDY as S

panel = est.load_panel(S)
print("panel:", panel.shape)

panel: (1117870, 27)


In [2]:
panel["D_q20"].eq(1).sum()

np.int64(58381)

## Magnitude sample — condition on existence

`build_magnitude_sample` keeps `D == 1` rows, forms `log_gap` / `log1p_gap_lag`, and drops pairs
with too few observations.

In [5]:
mag = est.build_magnitude_sample(panel, quantile=0.2,  min_obs_per_pair=S.min_events_per_pair)

magnitude: dropping 4 pairs: ['uniswap_2_vs_uniswap_4', 'uniswap_3_vs_uniswap_6', 'uniswap_4_vs_uniswap_6', 'uniswap_6_vs_pancake_1']
magnitude: (58261, 17) | mean log_gap: 0.8604 | pairs: 22


## Within pool-pair FE OLS

Entity-demeaned estimator (`linearmodels` PanelOLS), so the coefficient vector is just the eight
covariates. Two-way clustered (pool pair × block). In logs, each coefficient is a semi-elasticity
of the gap.

In [6]:
res = est.fit_magnitude_panel_ols(mag, two_way=True)
print(res)

                          PanelOLS Estimation Summary                           
Dep. Variable:                log_gap   R-squared:                        0.1796
Estimator:                   PanelOLS   R-squared (Between):              0.6846
No. Observations:               58261   R-squared (Within):               0.1796
Date:                Thu, Aug 20 2026   R-squared (Overall):              0.1962
Time:                        00:06:10   Log-likelihood                -9.491e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      1821.4
Entities:                          22   P-value                           0.0000
Avg Obs:                       2648.2   Distribution:                 F(7,58232)
Min Obs:                       57.000                                           
Max Obs:                    1.664e+04   F-statistic (robust):             363.24
                            